# Travel ML Project — Complete Notebook
## End-to-End Machine Learning: Regression · Classification · Recommendation

This notebook walks through the **complete ML pipeline** step by step:
1. Data loading & exploration
2. Data cleaning & feature engineering
3. Regression model (flight price prediction)
4. Classification model (gender prediction)
5. Recommendation system (hotel recommendation)
6. MLflow experiment tracking
7. API usage examples

## Step 1: Setup & Imports

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, classification_report
from xgboost import XGBRegressor

from src.data_processing import *
from src.utils import regression_metrics, classification_metrics

sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (12, 5)
print('Setup complete')

## Step 2: Load Datasets

In [ ]:
flights, hotels, users = load_datasets('../data')

print('FLIGHTS:', flights.shape)
display(flights.head(3))
print('\nHOTELS:', hotels.shape)
display(hotels.head(3))
print('\nUSERS:', users.shape)
display(users.head(3))

## Step 3: Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Price distribution by flight type
for ft in flights['flightType'].unique():
    flights[flights['flightType']==ft]['price'].plot(kind='hist', alpha=0.6, ax=axes[0], label=ft, bins=40)
axes[0].set_title('Flight Price Distribution by Type')
axes[0].set_xlabel('Price (USD)')
axes[0].legend()

# Price vs Distance scatter
sample = flights.sample(3000, random_state=42)
colors = {'firstClass': 'red', 'economic': 'blue', 'premium': 'green'}
for ft, grp in sample.groupby('flightType'):
    axes[1].scatter(grp['distance'], grp['price'], alpha=0.4, s=10, label=ft, color=colors[ft])
axes[1].set_title('Price vs Distance')
axes[1].set_xlabel('Distance (km)')
axes[1].set_ylabel('Price (USD)')
axes[1].legend()

# Gender distribution
users['gender'].value_counts().plot(kind='bar', ax=axes[2], color=['steelblue','coral','gray'])
axes[2].set_title('User Gender Distribution')
axes[2].set_xlabel('Gender')
axes[2].set_ylabel('Count')
axes[2].tick_params(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for flights
numeric_cols = ['price', 'time', 'distance']
corr = flights[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0)
plt.title('Flight Features Correlation')
plt.show()

# Average price by agency and flight type
pivot = flights.pivot_table(values='price', index='agency', columns='flightType', aggfunc='mean')
pivot.plot(kind='bar', figsize=(10,4))
plt.title('Average Price by Agency × Flight Type')
plt.ylabel('Avg Price (USD)')
plt.xticks(rotation=0)
plt.legend(title='Flight Type')
plt.tight_layout()
plt.show()

## Step 4: Data Processing Pipeline

In [ ]:
# Full regression data preparation
X_train, X_test, y_train, y_test, encoders, scaler = prepare_flight_regression_data(flights)

print(f'Train size : {X_train.shape}')
print(f'Test size  : {X_test.shape}')
print(f'\nFeatures: {list(X_train.columns)}')
print(f'\nTarget (price) stats:')
print(y_train.describe())

## Step 5: Regression Models — Flight Price Prediction

In [ ]:
mlflow.set_tracking_uri('file://../mlruns')
mlflow.set_experiment('notebook_flight_regression')

results = {}

# Linear Regression
with mlflow.start_run(run_name='LR_notebook'):
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    m = regression_metrics(y_test, lr.predict(X_test), 'LinearRegression')
    mlflow.log_metrics(m)
    results['Linear Regression'] = m

# Random Forest
with mlflow.start_run(run_name='RF_notebook'):
    rf = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    m = regression_metrics(y_test, rf.predict(X_test), 'RandomForest')
    mlflow.log_metrics(m)
    results['Random Forest'] = m

# XGBoost
with mlflow.start_run(run_name='XGB_notebook'):
    xgb = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, verbosity=0)
    xgb.fit(X_train, y_train)
    m = regression_metrics(y_test, xgb.predict(X_test), 'XGBoost')
    mlflow.log_metrics(m)
    results['XGBoost'] = m

res_df = pd.DataFrame(results).T
print('\n=== REGRESSION MODEL COMPARISON ===')
display(res_df.style.highlight_max(color='lightgreen').format('{:.4f}'))

In [ ]:
# Feature Importance (Random Forest)
fi = pd.DataFrame({'feature': X_train.columns, 'importance': rf.feature_importances_})
fi = fi.sort_values('importance', ascending=True)

fi.plot(kind='barh', x='feature', y='importance', legend=False, figsize=(10,5))
plt.title('Random Forest — Feature Importances (Flight Price)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

# Actual vs Predicted plot (XGBoost)
y_pred_xgb = xgb.predict(X_test)
plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred_xgb, alpha=0.3, s=5, color='steelblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect fit')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('XGBoost — Actual vs Predicted Flight Price')
plt.legend()
plt.tight_layout()
plt.show()

## Step 6: Classification — Gender Prediction

In [ ]:
X_tr2, X_te2, y_tr2, y_te2, enc2, sc2 = prepare_classification_data(users, flights, hotels)
print(f'Train: {X_tr2.shape} | Test: {X_te2.shape}')
print(f'Class balance: {pd.Series(y_tr2).value_counts().to_dict()}')
print(f'Features: {list(X_tr2.columns)}')

mlflow.set_experiment('notebook_gender_classification')
clf_results = {}

for name, clf in [
    ('Logistic Regression', LogisticRegression(C=1.0, max_iter=500, random_state=42)),
    ('Random Forest',       RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42))
]:
    with mlflow.start_run(run_name=name):
        clf.fit(X_tr2, y_tr2)
        pred = clf.predict(X_te2)
        m = classification_metrics(y_te2, pred, name)
        mlflow.log_metrics(m)
        clf_results[name] = m

clf_df = pd.DataFrame(clf_results).T
print('\n=== CLASSIFICATION MODEL COMPARISON ===')
display(clf_df.style.highlight_max(color='lightgreen').format('{:.4f}'))

## Step 7: Recommendation System

In [ ]:
from src.utils import load_artifact

rec = load_artifact('hotel_recommender')
print('Recommender loaded.')
print(f'Users: {len(rec.user_list)} | Hotels: {len(rec.hotel_list)}')

print('\n--- Collaborative Filtering (User 10) ---')
for r in rec.recommend_collaborative(10, top_n=5):
    print(f'  {r["hotel_name"]:12s} | score={r["score"]:.4f} | avg_price=${r["avg_price"]}')

print('\n--- Content-Based (Similar to Hotel A) ---')
for r in rec.recommend_content_based('Hotel A', top_n=5):
    print(f'  {r["hotel_name"]:12s} | similarity={r["similarity"]:.4f} | avg_price=${r["avg_price"]}')

print('\n--- Hybrid (User 10) ---')
for r in rec.recommend_hybrid(10, top_n=5):
    print(f'  {r["hotel_name"]:12s} | method={r["method"]}')

In [ ]:
# Visualise user-item matrix density
uim = rec.user_item_matrix
density = (uim > 0).sum().sum() / uim.size
print(f'User-Item Matrix: {uim.shape[0]} users × {uim.shape[1]} hotels')
print(f'Matrix density: {density:.2%}')

# Hotel booking frequency
booking_freq = uim.sum(axis=0).sort_values(ascending=False)
booking_freq.plot(kind='bar', figsize=(10,4), color='teal')
plt.title('Total Bookings per Hotel (User-Item Matrix)')
plt.ylabel('Total Bookings')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Step 8: API Usage Examples (Simulated)

In [ ]:
import requests, json

API_URL = 'http://localhost:5000'

# Example 1: Predict flight price
payload = {
    'from': 'Recife (PE)', 'to': 'Florianopolis (SC)',
    'flightType': 'economic', 'agency': 'CloudFy',
    'time': 2.0, 'distance': 900.0,
    'month': 6, 'dayofweek': 1, 'is_weekend': 0
}

try:
    resp = requests.post(f'{API_URL}/predict-flight-price', json=payload, timeout=5)
    print('Flight Price:', resp.json())
except Exception as e:
    print(f'API not running locally. Start with: python api/app.py')
    print(f'Error: {e}')

## Summary

| Task | Best Model | Key Metric |
|---|---|---|
| Flight Price Regression | Random Forest / XGBoost | R² ≈ 1.0 |
| Gender Classification | Random Forest | F1 ≈ 0.51 |
| Hotel Recommendation | Hybrid CF+CB | Density ≈ 82% |

**Next steps:**
- Run `python api/app.py` to start the REST API
- Run `streamlit run streamlit_app/app.py` for the interactive UI
- Run `mlflow ui --backend-store-uri file://./mlruns` to view experiment tracking
- Follow `README.md` for Docker and Kubernetes deployment